In [1]:
!pip install -q datasets

In [2]:
from datasets import load_dataset

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [4]:
ds["train"][0]

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
 'label': 1}

In [5]:
for i in range(5):
    print(ds["train"][i])
    print()

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}

{'text': 'the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson\'s expanded vision of j . r . r . tolkien\'s middle-earth .', 'label': 1}

{'text': 'effective but too-tepid biopic', 'label': 1}

{'text': 'if you sometimes like to go to the movies to have fun , wasabi is a good place to start .', 'label': 1}

{'text': "emerges as something rare , an issue movie that's so honest and keenly observed that it doesn't feel like one .", 'label': 1}



In [6]:
ds["train"]["label"][:10]

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [7]:
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

In [8]:
train_df.head()

,text,label
0,the rock is destined to be the 21st century's ...,1
1,"the gorgeously elaborate continuation of "" the...",1
2,effective but too-tepid biopic,1
3,if you sometimes like to go to the movies to h...,1
4,"emerges as something rare , an issue movie tha...",1


In [9]:
train_df['label'].value_counts()

label
1    4265
0    4265
Name: count, dtype: int64

In [10]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8530 entries, 0 to 8529
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    8530 non-null   object
 1   label   8530 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 133.4+ KB


In [11]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (8530, 2)
Validation: (1066, 2)
Test: (1066, 2)


In [12]:
from collections import Counter

Counter(ds["train"]["label"])

Counter({1: 4265, 0: 4265})

In [13]:
for i in range(5):
    print("Label:", ds["train"][i]["label"])
    print("Review:", ds["train"][i]["text"])
    print("-" * 80)

Label: 1
Review: the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .
--------------------------------------------------------------------------------
Label: 1
Review: the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth .
--------------------------------------------------------------------------------
Label: 1
Review: effective but too-tepid biopic
--------------------------------------------------------------------------------
Label: 1
Review: if you sometimes like to go to the movies to have fun , wasabi is a good place to start .
--------------------------------------------------------------------------------
Label: 1
Review: emerges as something rare , an issue movie that's so honest and k

In [14]:
train_df["text_length"] = train_df["text"].str.len()

train_df["text_length"].describe()

count    8530.000000
mean      113.971630
std        51.052231
min         4.000000
25%        76.000000
50%       111.000000
75%       149.000000
max       267.000000
Name: text_length, dtype: float64

In [15]:
train_df.groupby("label")["text_length"].mean()

label
0    112.772098
1    115.171161
Name: text_length, dtype: float64

### Step 4 — Text Preprocessing

In [16]:
import re
import string

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text

In [17]:
sample = ds["train"][0]["text"]

print("Before:")
print(sample)

print("\nAfter:")
print(preprocess_text(sample))

Before:
the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .

After:
the rock is destined to be the 21st centurys new  conan  and that hes going to make a splash even greater than arnold schwarzenegger  jeanclaud van damme or steven segal 


In [18]:
train_df["clean_text"] = train_df["text"].apply(preprocess_text)
val_df["clean_text"] = val_df["text"].apply(preprocess_text)
test_df["clean_text"] = test_df["text"].apply(preprocess_text)

In [19]:
train_df[["text", "clean_text", "label"]].head()

,text,clean_text,label
0,the rock is destined to be the 21st century's ...,the rock is destined to be the 21st centurys n...,1
1,"the gorgeously elaborate continuation of "" the...",the gorgeously elaborate continuation of the ...,1
2,effective but too-tepid biopic,effective but tootepid biopic,1
3,if you sometimes like to go to the movies to h...,if you sometimes like to go to the movies to h...,1
4,"emerges as something rare , an issue movie tha...",emerges as something rare an issue movie that...,1


### Step 5 — TF-IDF Vectorization

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train = tfidf.fit_transform(train_df["clean_text"])
X_val = tfidf.transform(val_df["clean_text"])
X_test = tfidf.transform(test_df["clean_text"])

In [21]:
y_train = train_df["label"]
y_val = val_df["label"]
y_test = test_df["label"]

In [22]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (8530, 18184)
X_val: (1066, 18184)
X_test: (1066, 18184)


In [23]:
print(tfidf.get_feature_names_out()[:20])

['00' '007' '10' '100' '100minute' '100year' '101' '102minute' '104' '105'
 '10course' '10inch' '10th' '10thgrade' '10yearold' '11' '110' '112minute'
 '117' '11th']


In [24]:
print(X_train[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 27 stored elements and shape (1, 18184)>
  Coords	Values
  (0, 16125)	0.09116471008841698
  (0, 13478)	0.19795084252240197
  (0, 8513)	0.06642350882376735
  (0, 4249)	0.2361864427981241
  (0, 16350)	0.12444524798413681
  (0, 1443)	0.10117621992589426
  (0, 108)	0.24067821609014067
  (0, 2548)	0.2726905130326547
  (0, 10716)	0.1439305864074776
  (0, 3218)	0.26087572671133735
  (0, 762)	0.05174566484497935
  (0, 16121)	0.0745901018457518
  (0, 7460)	0.17719828943544147
  (0, 6812)	0.16747252334973523
  (0, 9667)	0.13546022640722125
  (0, 15067)	0.24067821609014067
  (0, 5456)	0.12639642957028438
  (0, 6965)	0.23229549179026132
  (0, 16113)	0.11099682617099854
  (0, 997)	0.21814835314359862
  (0, 13873)	0.2257933456745122
  (0, 8616)	0.2726905130326547
  (0, 17253)	0.21814835314359862
  (0, 3853)	0.26087572671133735
  (0, 11168)	0.11874404045387313
  (0, 15292)	0.20559583505331555
  (0, 14011)	0.2726905130326547


### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_predict = model.predict(X_val)



accuracy = accuracy_score(y_val, y_predict)
precision = precision_score(y_val, y_predict)
recall = recall_score(y_val, y_predict)
f1 = f1_score(y_val, y_predict)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)


Accuracy : 0.7382739212007504
Precision: 0.7343173431734318
Recall   : 0.7467166979362101
F1 Score : 0.7404651162790697


### Step 8 — Word2Vec

In [29]:
!pip install -q gensim

In [30]:
from gensim.models import Word2Vec

train_tokens = train_df["clean_text"].str.split().tolist()
val_tokens = val_df["clean_text"].str.split().tolist()
test_tokens = test_df["clean_text"].str.split().tolist()

In [31]:
train_tokens[0]

['the',
 'rock',
 'is',
 'destined',
 'to',
 'be',
 'the',
 '21st',
 'centurys',
 'new',
 'conan',
 'and',
 'that',
 'hes',
 'going',
 'to',
 'make',
 'a',
 'splash',
 'even',
 'greater',
 'than',
 'arnold',
 'schwarzenegger',
 'jeanclaud',
 'van',
 'damme',
 'or',
 'steven',
 'segal']

In [32]:
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=42
)

In [33]:
w2v_model.wv["movie"]

array([-1.60213739e-01,  3.45794186e-02, -1.26739949e-01, -1.34414569e-01,
       -7.86509886e-02,  3.02505374e-01,  1.31292850e-01,  3.10227782e-01,
       -4.02437955e-01, -2.95622855e-01, -3.87146711e-01,  7.90590271e-02,
        3.15599024e-01,  2.64847159e-01,  1.26013950e-01,  1.10280685e-01,
        3.77818085e-02, -5.98133564e-01, -2.91216727e-02,  2.14718878e-01,
       -3.60259950e-01, -1.14398658e-01, -2.40164716e-03,  2.63027906e-01,
        4.00002480e-01,  8.65466446e-02,  6.32141680e-02,  2.06960887e-02,
       -3.65847141e-01,  9.41276476e-02, -5.15211448e-02, -3.12145799e-01,
        4.09833454e-02,  1.84146926e-01,  2.50131607e-01, -3.74035574e-02,
        4.10605639e-01, -3.76810580e-01, -2.16135427e-01,  1.73386171e-01,
        1.63038686e-01, -1.22762993e-01,  4.05073017e-01,  1.79465842e-02,
        1.12065524e-01,  5.71971089e-02,  2.53655255e-01,  2.15991452e-01,
        3.51567805e-01, -1.53084099e-01, -1.07462481e-01, -3.19177985e-01,
       -8.99341851e-02, -

In [34]:
w2v_model.wv["movie"].shape

(100,)

In [35]:
w2v_model.wv.most_similar("movie", topn=10)

[('film', 0.9519643783569336),
 ('documentary', 0.9001147747039795),
 ('picture', 0.9000074863433838),
 ('thing', 0.8846651911735535),
 ('entertainment', 0.8780398368835449),
 ('premise', 0.8743208646774292),
 ('seeing', 0.8731161952018738),
 ('certainly', 0.8724360466003418),
 ('worth', 0.8723759651184082),
 ('flick', 0.8721833229064941)]

In [36]:
import numpy as np

def review_to_vector(tokens, model):
    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [37]:
review_vector = review_to_vector(train_tokens[0], w2v_model)

print(review_vector)
print(review_vector.shape)

[-0.07637426  0.0791519  -0.07620422  0.02080978 -0.08673048  0.19475886
 -0.05678941  0.14959107 -0.18342412 -0.24136622 -0.32468352  0.1717455
  0.02568927  0.2727774   0.10393385 -0.02923321  0.07972092 -0.33393952
 -0.08347964  0.12548171 -0.10736751  0.01106632  0.02244544  0.31295565
  0.29353154  0.03870158 -0.01735982  0.16070664 -0.17812227  0.04079209
 -0.1690287  -0.12912601 -0.08172015  0.01743021  0.09600022 -0.10426304
  0.319725   -0.21679418 -0.12490198  0.17935899  0.08228954 -0.06612457
  0.26848456 -0.07481365  0.12482284  0.05547908  0.02574125  0.17720385
  0.3349113  -0.19031303  0.05567179 -0.38235724 -0.03703271 -0.16370055
  0.15136267  0.10948312 -0.09755429  0.14111839 -0.02206057  0.25032914
  0.14044923  0.15680237 -0.11832032  0.23184112  0.13778462  0.11421736
  0.13099538 -0.08195755 -0.16703883  0.04990772 -0.01244675 -0.02896936
 -0.04491279 -0.24413805  0.16344127 -0.06383482  0.0991874  -0.20250314
 -0.02007559 -0.19947767  0.00283961  0.07124738 -0.

In [38]:
X_train_w2v = np.array([
    review_to_vector(tokens, w2v_model)
    for tokens in train_tokens
])

X_val_w2v = np.array([
    review_to_vector(tokens, w2v_model)
    for tokens in val_tokens
])

X_test_w2v = np.array([
    review_to_vector(tokens, w2v_model)
    for tokens in test_tokens
])

In [39]:
print(X_train_w2v.shape)
print(X_val_w2v.shape)
print(X_test_w2v.shape)

(8530, 100)
(1066, 100)
(1066, 100)


In [40]:
w2v_classifier = LogisticRegression(random_state=42)

w2v_classifier.fit(X_train_w2v, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [41]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_val_pred_w2v = w2v_classifier.predict(X_val_w2v)


print("Accuracy :", accuracy_score(y_val, y_val_pred_w2v))
print("Precision:", precision_score(y_val, y_val_pred_w2v))
print("Recall   :", recall_score(y_val, y_val_pred_w2v))
print("F1 Score :", f1_score(y_val, y_val_pred_w2v))

Accuracy : 0.5984990619136961
Precision: 0.5922671353251318
Recall   : 0.6322701688555347
F1 Score : 0.6116152450090744


### Step 9 — BERT

In [42]:
!pip install -q transformers accelerate

In [43]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [45]:
text = "This movie was absolutely amazing!"

tokens = tokenizer(text)

print(tokens)

{'input_ids': [101, 2023, 3185, 2001, 7078, 6429, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [46]:
tokenizer.tokenize("This movie was absolutely amazing!")

['this', 'movie', 'was', 'absolutely', 'amazing', '!']

In [47]:
tokenizer.tokenize("unbelievable")

['unbelievable']

### Step 10 — Prepare data for BERT

In [48]:
from datasets import Dataset

train_hf = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_hf = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_hf = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

In [49]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [50]:
train_tokenized = train_hf.map(tokenize_function, batched=True)
val_tokenized = val_hf.map(tokenize_function, batched=True)
test_tokenized = test_hf.map(tokenize_function, batched=True)

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [51]:
from transformers import AutoModelForSequenceClassification

bert_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [52]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [53]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [54]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

In [55]:
from transformers import Trainer

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [56]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.evaluate()